In [6]:
from Z import is_prime, divs, prime_power, factor, gcd, lcm, phi
from functools import total_ordering
from nimber_ops import *
from transfinite import Ordinal
import numpy as np
import pandas as pd

In [7]:
W = Ordinal

In [8]:
# utility functions: collect into util.py later
def base(n : int, b : int, length :  int = 0) -> list[int]:
    ''' returns the base b expansion of n as a  list
    n = sum_j^N a_j*b^j returns [a_0, a_1, ..., a_N]
    optional length parameter to include leading zeros if desired,
    otherwise minimum possible length where highest digit is non-zero'''
    assert (n >= 0 and b > 1), ('positional base expansion defined for' 
                                'non-negative integers and positive bases')
    if length == 0:
        if n == 0: return [0]
        coeffs = []
        while n > 0:
            coeffs.append(n % b)
            n //= b
        return coeffs
    else:
        coeffs = base(n, b)
        while len(coeffs) < length:
            coeffs.append(0)
        return coeffs

def base_eval(coeffs : list[int], b : int) -> int:
    total = 0
    for coeff in coeffs[::-1]:
        total = coeff + b * total
    return total
  
def ord_decomp(ordinal : Ordinal | int) -> list:
    ''' returns [inf_1, inf_2, ..., inf_n, finite term]'''
    if isinstance(ordinal, int):
        assert ordinal >= 0
        return [ordinal]
    high = Ordinal(ordinal.exponent, ordinal.coefficient)
    terms = [high]
    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        terms.append(Ordinal(remainder.exponent, remainder.coefficient, 0))
        remainder = remainder.addend
    assert(isinstance(remainder, int))
    terms.append(remainder)
    return terms

def ord_recomp(list : list) -> Ordinal | int:
    result = 0
    terms = sorted(list, reverse=True)
    for term in list:
        result = result + term
    return result

def ord_subtract(a, b):
    """
    Given ordinals a > b, return the ordinal c such that b + c == a

    """
    if a ==  b: return 0
    
    if a < b:
        raise ValueError("First argument must be greater than second argument")

    if type(a) == int:
        return a - b

    if type(b) == int or a.exponent > b.exponent:
        return a

    if a.exponent == b.exponent and a.coefficient == b.coefficient:
        return ord_subtract(a.addend, b.addend)

    # Here we know that a.coefficient > b.coefficient
    return Ordinal(a.exponent, a.coefficient - b.coefficient, a.addend)

def ord_div(ord1 : Ordinal, ord2 : Ordinal):
    ''' assuming both are single term: (w^a)*c / w^b = (w^(a - b))*c'''
    assert ord1 >= ord2
    if ord2 == 1: return ord1
    ex1, ex2 = ord1.exponent, ord2.exponent
    c = ord1.coefficient
    return Ordinal(ord_subtract(ex1, ex2), c)

def int_log(N: int, base: int) -> int:
    ''' Returns p such that base ^ p <= N < base ^ (p+1)'''
    def level(N: int, p: int = 2) -> int:
        '''
        Returns the largest 'level' L (w.r.t. p) such that 
        p ** (2 ** L) <= N
        '''
        L = 0
        if N < p:
            return 0
        while N // (p ** (1 << L)) != 0:
            L += 1
        return L-1

    if N < base:
        return 0
    # for the highest power 'exp' s.t. base^exp <= N, find
    # the largest power of 2 that is less than or equal to exp
    total = 1 << level(N, base)
    # now divide N to recursively find the binary expansion of exp
    N = N // (base ** total)
    while N != 0:
        total += 1 << level(N, base)
        N = N // (base ** (1 << level(N, base)))
    return total - 1

def pi(n : int) -> int:
    ''' prime counting function: returns # primes < n'''
    with open('small_primes.txt') as file:
        num_less = 0
        for line in file:
            p = int(line)
            if p < n:
                num_less += 1
            else:
                return num_less
        raise ValueError('n is too big to calculate pi(n) by brute force')
            
            
def f(p):
    '''Lenstra's f function: 
    for prime p, f(p) is min{ n | p divides 2^n-1}
    It is always the case that f(p) divides p-1'''
    divisors = divs(phi(p))
    for div in divisors[:-1]:
        if ((1<<div) -1) % p == 0:
            return div
    return divisors[-1]


small_primes = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,\
    83,89,97,101,103,107,109,113,127,131,137,139,149,151,157,163,167,173,179,181,191,193,197,199,211,223,227,229,233,239,241,251,257,263,269,271,277,281,283,293,307,311,313,317,331,337,347,349,353,359,367,373,379,383,389,397,401,409,419,421,431,433,439,443,449,457,461,463,467,479,487,491,499,503,509,521,523,541,547,557,563,569,571,577]

df = pd.read_csv('alpha_p.csv') # got them all calculated as high as possible!
w = Ordinal()
alpha_p = {}
for i in df.index:
    alpha_p[int(df.loc[i, 'p'])] = eval(df.loc[i, 'alpha_p'])

####### this part was for the inductive construction of alpha_p ##############
###############################################################################
# kappa_p = { # generator of smallest extension of prime degree p
#            # kappa_{p_^n} = w^{w^{k-1}*p^{n-1}} where k = |{primes < p}|
#            2: 2, 3 : Ordinal()}
# for i, prime in enumerate(small_primes[2:]):
#     kappa_p[prime] = Ordinal(Ordinal(i+1))
# alpha_p = { # (kappa_p)^p, e.g. w^3=2, [w^w]^5 = 4, [w^w^2]^7 = w+1, etc.
#            # very hard to calculate for higher p, based on http://www.neverendingbooks.org/on2-extending-lenstras-list/
#             2:3, 3:2, 5:4, 7:Ordinal()+1, 11:Ordinal(Ordinal())+1, 13:Ordinal()+4,
#             17: 16, 19:Ordinal(3)+4, 23:Ordinal(Ordinal(3))+1, 
#             29:Ordinal(Ordinal(2))+4, 31:Ordinal(Ordinal())+1, 37: Ordinal(3)+4,
#             41: Ordinal(Ordinal())+1, 43:Ordinal(Ordinal(2))+1, 
#             47:Ordinal(Ordinal(7))+1, 
#             53: Ordinal(Ordinal(4))+1, 
#             59:Ordinal(Ordinal(8))+1, 
#             61:Ordinal(Ordinal())+Ordinal(), 
#             67:Ordinal(Ordinal(3))+Ordinal(),
#             71:Ordinal(Ordinal(2))+Ordinal(Ordinal()),
#             73:Ordinal(3)+1,
#             79:Ordinal(Ordinal(4))+1,
#             83:Ordinal(Ordinal(11))+1,
#             89:Ordinal(Ordinal(3))+1,
#             97:Ordinal()+256,
#             101:Ordinal(Ordinal()*5)+1,
#             103:Ordinal(Ordinal(5))+Ordinal()}
# excess = [3,0,0,1,1,0,0,4,1,0,1,0,1,1,1,1,1,0,0,0,1,1,1,1,0,
#  1,0,1,0,0,1,0,1,0,1,0,1,4,1,0,1,0,0,0,0,0,0,1,1,1,
#  1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,0,
#  1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,0,0,0,
#  0,1,1,1,1,0] # use https://oeis.org/A380496/

In [17]:
@total_ordering
class Nim:
    """nimbers"""

    def __init__(self, n: int | Ordinal) -> None:
        """ordinal considered an a field element in On_2
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        """

        def exp2(level: int) -> int:
            return 1 << level

        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True
            if self.val < 2:
                self.field = 2  # smallest
                self.base = 0
                self.high = 0
                self.low = self.val
                self.level = 0
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.level = level
                self.field = exp2(exp2(level))
                self.base = exp2(exp2(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            assert n < Ordinal(Ordinal(Ordinal())), 'ordinals < omega^omega^omega are implemented'
            self.val = n
            self.isfinite = False
            # these attributes can be found, but not as useful for infinite nums
            self.field = None
            self.base = None
            self.high = None
            self.low = None
            # assert isinstance(
            #     n, Ordinal), 'An infinite nimber must be an ordinal'
            # self.val = n
            # self.isfinite = False
            # # find the smallest field containing n
            # if (k := n.exponent) < Ordinal():  # n = omega^k, k fintie => cubic extension
            #     power = 0
            #     while 3 ** power <= k:
            #         power += 1
            #     self.field = Ordinal(3**power)
            #     exp = 3**(power-1)
            #     self.base = Ordinal(exp)
            #     high = Ordinal(n.exponent - exp, n.coefficient,
            #                 0) if exp < n.exponent else n.coefficient
            #     remainder = n.addend
            #     while isinstance(remainder, Ordinal) and remainder.exponent > exp:
            #         high += Ordinal(remainder.exponent - exp,
            #                         remainder.coefficient, 0)
            #         remainder = remainder.addend
            #         if isinstance(remainder, Ordinal) and remainder.exponent == exp:
            #             high += remainder.coefficient
            #             remainder = remainder.addend
            #     self.high = high
            #     self.low = remainder

    def __add__(self, other):
        if self.isfinite and other.isfinite:
            return Nim(self.val ^ other.val)  # finite nim sum is bitwise XOR
        if self.val == other.val:
            return Nim(0)
        ord1, ord2 = self.val, other.val
        terms1, terms2 = ord_decomp(ord1), ord_decomp(ord2)
        sum = {}
        for term in terms1[:-1]:
            sum[term.exponent] = term.coefficient
        for term in terms2[:-1]:  # nim sum the coeffs if any terms with same exp
            try:
                sum[term.exponent] = sum[term.exponent] ^ term.coefficient
            except:
                sum[term.exponent] = term.coefficient

        keys = sorted(sum.keys())  # ordinal addition not commutative
        result = terms1[-1] ^ terms2[-1]  # nim sum of finite part
        for key in keys:
            if sum[key] > 0:  # add bigger on the left
                result = Ordinal(key, sum[key]) + result
        return Nim(result)

    def __eq__(self, other):
        return self.val == other.val

    def __hash__(self):
        return self.val.__hash__()

    def __lt__(self, other):
        return self.val < other.val

    def __mul__(self, other):
        assert isinstance(other, Nim)
        x, y = self.val, other.val
        if x == 0 or y == 0:
            return Nim(0)
        if x == 1:
            return other
        if y == 1:
            return self
        if self.isfinite and other.isfinite:
            if self.val == other.val:
                return self.sq()

            def nim_product(a: int, b: int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low

                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a, q_b) * F_b ^ nim_product(a, r_b)
                    elif F_a > F_b:
                        return nim_product(q_a, b) * F_a ^ nim_product(r_a, b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a, q_b)
                        p_2 = nim_product(r_a, r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4

            return Nim(nim_product(x, y))
        elif self.isfinite and not other.isfinite:
            terms = ord_decomp(other.val)  # distribute to each coefficient
            terms[-1] = (self * Nim(terms[-1])).val
            for term in terms[:-1]:
                term.coefficient = (self * Nim(term.coefficient)).val
            return Nim(ord_recomp(terms))
        elif not self.isfinite and other.isfinite:
            return other * self
        else:  # both infinite
            terms1 = ord_decomp(self.val)
            if self.val == other.val and (len(terms1) > 2 or terms1[-1] > 0):
                return self.sq()
            terms2 = ord_decomp(other.val)
            inf1, fin1 = terms1[:-1], terms1[-1]
            inf2, fin2 = terms2[:-1], terms2[-1]
            if fin1 == 0 and fin2 == 0:
                to_sum = {Nim(0)}
                # result = Nim(0)

            else:
                # start by "FOIL-ing" to handle the terms where one is finite
                to_sum = (
                    {Nim(fin1) * other} ^ {self * Nim(fin2)} ^ {Nim(fin1) * Nim(fin2)}
                )
                # result = Nim(fin1) * other + self * Nim(fin2) \
                # + Nim(fin1) * Nim(fin2)
            for x in inf1:  # expand and distribute the purely infinite terms
                for y in inf2:
                    # calculate (w^N * a) x (w^M * b)
                    X, Y = sorted([x, y], reverse=True)  # X >= Y
                    N, M = X.exponent, Y.exponent  # N >= M
                    a, b = X.coefficient, Y.coefficient
                    coeff = Nim(a) * Nim(b)

                    if N < Ordinal() and M < Ordinal():  # handle finite case
                        N_tern = base(N, 3)  # write exponents in ternary
                        M_tern = base(M, 3)
                        K = max([len(N_tern), len(M_tern)])

                        # invert the powers of 3 since w^(3^k) = 2^(3^{-k-1})
                        def phi(n):
                            return base_eval(base(n, 3, K)[::-1], 3)  # reversed 3s

                        exp_p = phi(N) + phi(M)  # the exponent 2^(3^{-K} * exp_2)

                        next_power = 3**K
                        q, r = exp_p // next_power, exp_p % next_power
                        coeff = coeff * Nim(2) ** q
                        W = Nim(Ordinal(phi(r))) if r > 0 else Nim(1)
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    elif N < Ordinal() and not M < Ordinal():
                        W = Nim(Ordinal(N + M))
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    else:
                        assert max(N, M) < Ordinal(
                            105
                        ), "Nimber multiplication is only implemented for ordinals < w**w**105"
                        N_decomp, M_decomp = ord_decomp(N), ord_decomp(M)
                        # multiply all omega terms using exponent rules
                        N_fin_exp, M_fin_exp = N_decomp[-1], M_decomp[-1]
                        N_inf_exp, M_inf_exp = N_decomp[:-1], M_decomp[:-1]
                        # handle finite exponent first
                        omega_N = Ordinal(N_fin_exp) if N_fin_exp != 0 else 1
                        omega_M = Ordinal(M_fin_exp) if M_fin_exp != 0 else 1
                        # start multiplying omegas
                        prod: Ordinal = (Nim(omega_N) * Nim(omega_M)).val
                        # group the exponents by like terms
                        N_dict = {exp.exponent: exp for exp in N_inf_exp}
                        M_dict = {exp.exponent: exp for exp in M_inf_exp}
                        N_exp_exp = set(N_dict)
                        M_exp_exp = set(M_dict)
                        like_terms = N_exp_exp & M_exp_exp
                        unlike_terms = (N_exp_exp | M_exp_exp) - like_terms
                        for t in sorted(unlike_terms):  # normal ordinal product
                            mult = N_dict[t] if t in N_dict else M_dict[t]
                            prod = Ordinal(mult) * prod  # pull through
                        for n in like_terms:
                            i = N_dict[n].coefficient
                            j = M_dict[n].coefficient
                            p = small_primes[n + 1]
                            alpha = Nim(alpha_p[p])
                            i_p = base(i, p)  # write coeff in base p
                            j_p = base(j, p)
                            K = max([len(i_p), len(j_p)])

                            # invert the powers of p
                            def phi(n):
                                return base_eval(base(n, p, K)[::-1], p)

                            exp_p = phi(i) + phi(j)

                            next_power = p**K
                            q, r = exp_p // next_power, exp_p % next_power
                            coeff = coeff * alpha**q
                            term = Ordinal(Ordinal(n) * phi(r)) if r > 0 else 1
                            # need recursive because might be new like terms
                            prod = (Nim(prod) * Nim(term)).val
                        to_sum ^= {coeff * Nim(prod)}
                        # result = result + coeff * Nim(prod)
            return Nim(0).sum(*to_sum)

    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if p >= 0:  # binary exponentiation by squaring
            result = Nim(1)
            nimber = self
            while p > 0:
                if p & 1:
                    result = result * nimber
                nimber = nimber.sq()
                p >>= 1
            return result
        elif p == -1:
            if self.isfinite:
                if self.field == 2:
                    assert self.val > 0, 'inverse is not defined for zero'
                    return self
                a, b, F, f = (
                    Nim(self.high),
                    Nim(self.low),
                    Nim(self.base),
                    Nim(self.base >> 1),
                )
                det = (a + b) * b + a * a * f
                return det ** (-1) * (a * F + (a + b))
            # for infinite we need to invert via multiplication matrix 
            zero = Nim(0)
            one = Nim(1)

            # 2. Create the Augmented Matrix [A | e_0]
            A = np.array(self.matrix(), dtype=object)
            n = A.shape[0]
            
            b_col_list = [zero] * n
            b_col_list[0] = one
            b = np.array(b_col_list, dtype=object).reshape(n, 1) 
            
            Aug = np.concatenate((A, b), axis=1)
            
            # 3. Forward Elimination
            for k in range(n):
                # --- Pivoting ---
                pivot = Aug[k, k]
                
                if pivot == zero:
                    for i in range(k + 1, n):
                        if Aug[i, k] != zero:
                            Aug[[k, i]] = Aug[[i, k]]
                            break
                    else:
                        # Singular Matrix
                        return None 

                # --- Elimination ---
                pivot = Aug[k, k] # Re-get pivot in case of swap
                for i in range(k + 1, n):
                    if Aug[i, k] == zero:
                        continue
                    
                    factor = Aug[i, k] / pivot
                    
                    # **Single Vectorized Operation**
                    Aug[i, k:] = Aug[i, k:] + Aug[k, k:] * factor

            # 4. Back Substitution
            x = np.array([zero] * n, dtype=object)

            for i in range(n - 1, -1, -1):
                # Sum of U[i, j] * x[j] for j > i
                if i < n - 1:
                    sum_ax = np.dot(Aug[i, i+1:n], x[i+1:])
                    val = Aug[i, n] + sum_ax 
                else:
                    val = Aug[i, n] # For the last row
                    
                x[i] = val / Aug[i, i]
            baseF = Nim(self.base_field())
            basis = np.array([baseF**k for k in range(n)], dtype=object)
            return np.dot(x, basis)
        else:
            inv = self ** (-1)
            return inv ** (-p)

    def __repr__(self) -> str:
        if self.isfinite:
            return str(self.val)
        else:
            return self.val.__repr__()

    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()

    def __truediv__(self, other):
        if not isinstance(other, int|Ordinal|Nim):
            raise ValueError('Quotient of nimbers must be be both nimbers')
        if isinstance(other, int|Ordinal):
            div = Nim(other)
        else:
            div = other
        assert div.val != 0, 'division by zero is not defined!'
        return self * div ** -1

    def as_vec(self, as_nim=True, over=None, ret_basis=False) -> list|np.ndarray:
        ''' write the nimber as a vector n = a0*b0 + ... + aj*bj and return 
        [a0, ..., aj] where B = {b_j} is the basis {(nimber.base_field)^j}
        e.g.  7 = 3 + 1*4 returns [3, 1]
              omega^2 + omega*8 + 12 returns [12, 8, 1], etc. '''
        if over == None:
            if self.isfinite:
                return [Nim(self.low), Nim(self.high)] if as_nim else [self.low, self.high]
            baseF, nextF = self.field_interval()
            dim = (nextF.exponent // baseF.exponent if baseF < Ordinal(Ordinal()) 
                else nextF.exponent.coefficient // baseF.exponent.coefficient)
            terms = ord_decomp(self.val)
            vec = [0 for _ in range(dim)]
            def ord_div(ord1 : Ordinal, ord2 : Ordinal):
                ''' assuming both are single term: (w^a)*c / w^b = (w^(a - b))*c'''
                assert ord1 >= ord2
                if ord2 == 1: return ord1
                ex1, ex2 = ord1.exponent, ord2.exponent
                c = ord1.coefficient
                return (Ordinal(ord_subtract(ex1, ex2), c) if ord_subtract(ex1, ex2)>0 
                        else c)
            j = 0
            for term in terms[::-1]:
                if term == 0: continue
                while term >= baseF ** (j + 1):
                    j+=1
                vec[j] = ord_div(term, baseF**j) + vec[j]
            return vec if not as_nim else [Nim(j) for j in vec]
        elif over == 2:
            ''' the other extreme: rather than expanding over biggest possible
            subfield, we expand over the smallest possible subfield (namely, 2)'''
            dim = self.deg()
            baseF = kappa(dim)
            vec = np.full(dim, 0)
            basis = sorted([baseF**n for n in range(dim)], reverse=True)
            for i, b in enumerate(basis):
                if b <= self:
                    vec[i] += 1
                    self = self + b
            return [vec, basis] if ret_basis else vec
            
            
            
        else:
            raise ValueError(f'writing as a vector space over {over} isn\'t implemeted')
    def base_field(self) -> Ordinal:
        ''' returns the greatest ordinal F <= self.val which is a field 
        under Nim operations. e.g. base_field(3)=2, base_field(omega)=omega, etc.'''
        return self.field_interval()[0]

    def deg(self, added_one=False) -> int:
        """returns the degree of the minimal polynomial"""
        if self.isfinite:
            return self.field.bit_length() - 1
        # if infinite, the kappa numbers are easier to calculate degree
        if self.is_field():
            return kappa_deg(self.is_field())
        # try writing as a sum of kappa terms
        terms = ord_decomp(self.val)
        if terms[-1] == 0: terms.pop() # remove trailing zero
        kappa_nums = [Nim(term).is_field() for term in terms]
        if all(kappa_nums):
            return lcm(*[kappa_deg(num) for num in kappa_nums])
        # last thing to try is adding one, since this doesn't change degree
        if added_one == False:
            return (self + Nim(1)).deg(added_one=True) # hopefully easier form
        
        # otherwise, we have no choice but to brute force. Might be very slow!
        d = 1
        x = self * self
        while x != self:
            x = x * x
            d += 1
        return d

    def det(self):
        """det(N) = determinant of the multiplication by N matrix
        This is the same as the field norm over the next smallest field.
        For x < omega, this is equivalent to
        det(x) = x^(x.base + 1) = x^(2^2^n + 1)  if 2^2^n <= x < 2^2^(n+1)
        
        For x>= omega, there might not be a next smallest field, so there
        is more choice for which subfield to express as a linear trans over
        """
        if self.isfinite:
            if self.field == 2:
                return self
            a, b, F = Nim(self.high), Nim(self.low), Nim(self.base >> 1)
            return (a + b) * b + a.sq() * F  # much faster than self**(self.base+1)
        # else
        A = self.matrix()
        n = A.shape[0]
        zero = Nim(0)
        one = Nim(1)
        
        # Forward Elimination
        det = one
        for k in range(n):
            # --- Pivoting ---
            pivot = A[k, k]
            
            if pivot == zero:
                # Search for swap
                for i in range(k + 1, n):
                    if A[i, k] != zero:
                        A[[k, i]] = A[[i, k]] # Vectorized swap
                        break
                else:
                    return zero # Singular Matrix

            pivot = A[k, k]
            det = det * pivot # Accumulate determinant

            # --- Elimination ---
            for i in range(k + 1, n):
                if A[i, k] == zero:
                    continue
                
                factor = A[i, k] / pivot
                
                # **Vectorized row operation**
                A[i, k:] = A[i, k:] + A[k, k:] * factor
        return det

    def det_star(self):
        """number of times x -> det(x) is repeated until landing in F_2"""
        iter = 0
        if self.level == 0:
            return iter
        d = self.det()
        while d != Nim(1):
            iter += 1
            d = d.det()
        return iter + 1

    def field_interval(self) -> list[Ordinal|int]:
        """returns (base, next), where base <= nimber < next and both are fields
        e.g. 2 -> (2, 4), 7 -> (4, 16), omega -> (omega, omega^3), etc. """
        if self.isfinite:
            return [self.base, self.field]
        
        # otherwise, only need to consider the highest power of Omega
        lead_exponent = ord_decomp(self.val)[0].exponent
        if lead_exponent < Ordinal():
            # field(w**n) = w^(3^K) where 3^(K-1)<= n < 3^K
            power = int_log(lead_exponent, 3)
            baseF = Ordinal(3**power)
            nextF = Ordinal(3**(power + 1))
            return [baseF, nextF]
        
        # if leading exponent is itself infinite, we need leading power of *that*
        lead_lead = ord_decomp(lead_exponent)[0]
        coef, ex = lead_lead.coefficient, lead_lead.exponent
        # which prime base we interpret coef depends on the exponent
        p = small_primes[ex + 1]
        # want the smallest power of p exceeding coef
        power = int_log(coef, p)
        # next smallest field is omega^(p^(next_power)omega^ex)
        baseF = Ordinal(Ordinal(ex, p**power))
        nextF = Ordinal(Ordinal(ex, p**(power+1)))
        return [baseF, nextF] 
        
    def inv(self):
        return self ** (-1)

    def is_gen(self) -> bool:
        """True iff order(self) == self.field - 1"""
        if self.isfinite:
            """is_gen ==> det* = self.level
            det* = self.level ==> is_gen if level < 6"""
            if self.val == 0:
                return False
            if self.level != self.det_star():
                # is_gen ==> det* = self.level
                return False
            elif self.level < 6:
                # self.base = 2^2^(n-1) + 1 is prime for n < 6
                return True
            elif self.level == 6:  # F_5 = 641 × 6,700,417
                level_small = (self**641).level
                level_big = (self**6700417).level
                return min(level_small, level_big) == 6  # both are divisors of order
            elif self.level == 7:  # F_7 = 274,177 × 67,280,421,310,721
                level_small = (self**274177).level
                level_big = (self**67280421310721).level
                return min(level_small, level_big) == 7 and self.det().is_gen()
            # could keep going, but the highest factored is 'only' F_11 anyways
            else:
                return self.order() == (1 << (1 << self.level)) - 1

    def is_field(self) -> int:
        ''' returns 0 if the ordinal [x] isn't a field under nim operations,
            returns p^n if [x] = kappa_{p^n}
        '''
        F = self.base_field()
        if F != self.val: 
            return 0
        if F < Ordinal():
            return 2 * int_log(F, 2) # kappa_{2^n} = [2^2^{n-1}]
        if F.exponent < Ordinal():
            p = 3
            n = int_log(F.exponent, p) # next field is kappa_{p^n}
        else:
            p = small_primes[(F.exponent).exponent + 1]
            n = int_log(F.exponent.coefficient, p)
        return p**(n+1)
    
    def matrix(self) -> np.ndarray:
        '''' returns the multiplication map of the nimber considered as a linear
        transform over the field [self.base] '''
        if self.isfinite:
            a, b = Nim(self.low), Nim(self.high)
            mat =  [[a,  b ], 
                    [b, a+b]]
            return np.array(mat, dtype=object)
        # else, use the basis {1, base, base^2, ..., base^(d-1)}
        baseF, nextF = self.field_interval()
        dim = (nextF.exponent // baseF.exponent if baseF < Ordinal(Ordinal()) 
               else nextF.exponent.coefficient // baseF.exponent.coefficient)
        kapp = Nim(baseF)
        alph = kapp**dim
        curr_row = np.array(self.as_vec(), dtype=object)
        mat = [curr_row]
        for _ in range(dim - 1):
            curr_row = np.roll(mat[-1], 1)
            curr_row[0] = curr_row[0] * alph
            mat.append(curr_row)
        return np.array(mat, dtype=object).T

    def next_field(self) -> Ordinal:
        return self.field_interval()[1]
        
    def order(self) -> int:
        if self.isfinite:
            if self.level <= 7:
                if self.is_gen():  # avoid infinite loop
                    return (1 << (1 << self.level)) - 1

            n = self.val
            L = self.level
            if L == 0:
                return n
            elif L == 1:
                return 3  # ord(2) = ord(3) = 3
            elif L <= 5:  # order divides 3 * 5 * 17 * 257 * 65_537
                return (self.base + 1) * self.det().order()
            else:
                factor = 1
                nimber = self
                while nimber.level > 5:
                    nimber = nimber.det()
                    factor *= nimber.base + 1

                prime_divs = nimber.order()
                nimber_to_primes = self**prime_divs  # order divides F_5*...*F_{L-1}
                factor *= prime_divs
                max_non_gen = (1 << (1 << L) - 1) // 3
                print(
                    f"Warning: Order of {self} may take forever to calculate:\
                        too difficult to factor {factor}"
                )
                for i in range(641, max_non_gen // prime_divs + 1, 2):
                    if (factor // prime_divs) % i:
                        continue
                    if (nimber_to_primes**i).val == 1:
                        return i * prime_divs
                return 1 << (1 << L) - 1
                # # make more efficient by only checking possible orders
                # # use Lagrange's theorem
                # # find the smallest field containing n i.e. smallest F_k > n
                # exp = (n.bit_length() - 1).bit_length()
                # # find the order of n must divide F_k - 1 which factors by difference of squares
                # divisors = fermat_divisors(exp)
                # for factor in divisors[:-1]:
                #     if (Nim(n)**factor).val == 1:
                #         return factor
                # else:
                #     return divisors[-1]
        else:
            ...  # inifite case is hard..

    def sum(self, *args):
        if len(args) == 0:
            return self
        elif len(args) == 1:
            return self + args[0]
        else:
            *inf, fin = ord_decomp(self.val)
            terms = {term.exponent: term.coefficient for term in inf}
            terms[0] = fin
            for arg in args:
                *inf_, fin_ = ord_decomp(arg.val)
                terms[0] ^= fin_
                for term_ in inf_:
                    try:
                        terms[term_.exponent] = (
                            terms[term_.exponent] ^ term_.coefficient
                        )
                    except:
                        terms[term_.exponent] = term_.coefficient
            result = terms[0]
            for key in sorted(terms.keys())[1:]:
                if terms[key] > 0:
                    result = Ordinal(key, terms[key]) + result
            return Nim(result)

    def sq(self):
        """returns the Nimber's square using Freshman's Dream"""
        if self.isfinite:
            a, b, base = self.high, self.low, self.base
            # x = a *2^2^n + b
            # x^2 = (a^2)*(2^2^n + 2^(2^n-1)) + b^2
            if self.base == 0:  # either 0 or 1
                return self
            else:
                term = base + (base >> 1)
                return Nim(a).sq() * Nim(term) + Nim(b).sq()
        else:
            terms = ord_decomp(self.val)
            sum = Nim(terms[-1]).sq()
            for term in terms[:-1]:
                sum = sum + Nim(term) * Nim(term)
            return sum

    def sqrt(self):
        if self.isfinite:
            if self.field == 2:
                return self
            term = self.sq() + self
            return term.sqrt() + self
        else:
            F = self.next_field()
            if F.exponent < Ordinal():
                p = 3
                n = int_log(F.exponent, p) # next field is kappa_{p^n}
            else:
                p = small_primes[(F.exponent).exponent + 1]
                n = int_log(F.exponent.coefficient, p)
            degree = kappa_deg(p**n)
            # x^2^d = x ==> sqrt(x) = x^2^(d-1)
            return self ** (1 << (degree - 1)) 
        
#-------------------------------------------------------------------------------#

def kappa(h : int) -> Nim:
    assert h > 0
    if h == 1 : result = Nim(0)
    elif prime_power(h):
        p, n = prime_power(h)
        k = pi(p)
        exp = Ordinal(k, h//p) if k > 0 else h//p
        result = Nim(2**exp)
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = kappa(g)
        else:
            result = kappa(q) + kappa(g)
    return result

def alpha(p):
    return Nim(alpha_p[p])
      
def Q(h : int) -> list[int]:
    ''' kappa(h) = sum_{q in Q} kappa(q), q prime powers'''
    assert h > 0
    if h == 1 : 
        result = []
    elif prime_power(h):
        result = [h]
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = Q(g)
        else:
            result = Q(g) + Q(q)
    return result

def kappa_deg(h : int) -> int:
    ''' faster way to calculate degree for kappa_h'''
    if h == 1: result = 1    
    elif prime_power(h):
        p, n = prime_power(h)
        if p == 2: result = h # deg(k_{2^n}) = 2^n
        else:
            result = h * lcm(kappa_deg(f(p)), (Nim(alpha_p[p]) + kappa(f(p))).deg())
    else:
        result = lcm(*[kappa_deg(q) for q in Q(h)])
    return result

def enum_kappa_interval(p, from_kappa=False):
    ''' generator which iterates over smallest field containing kappa_{f(p)}
    
    option to start at kappa_{f(p)}'''
    k = kappa(f(p))
    d = kappa_deg(f(p))

    if from_kappa:
        basis = sorted([k**n for n in range(d)], key=lambda z:z.val)
        start_index = 0
        while basis[start_index].val < k.val:
            start_index += 1
    else: 
        basis = [k**n for n in range(d)]
        start_index = 0
    for i in range(1<<start_index, 1<<d):
        result = Nim(0)
        i_base2 = base(i, 2)
        i_bits = [j for j, bit in enumerate(i_base2) if bit==1]
        yield result.sum(*[basis[j] for j in i_bits])
        
def smallest_of_order(p):
    for nimber in enum_kappa_interval(p):
        if nimber**p==Nim(1) and nimber.val > 1:
            return sorted([nimber**k for k in range(1, p)], key = lambda x:x.val)[0]
        

In [30]:
def tr(nimber:Nim, over):
    deg = nimber.deg()
    assert deg % over == 0
    curr = nimber
    conjs = [nimber]
    for _ in range(1, deg):
        curr = curr ** (2**over)
        conjs.append(curr)
    return Nim(0).sum(*conjs)

In [38]:
x11= tr(Nim(W(W(3))), 11)

In [ ]:
sorted([x11**n for n in range(1, 2**11-1) if gcd(n, 2**11-1)[-1]==1])

In [32]:
def alpha_ord(p):
    alph = Nim(alpha_p[p])
    deg = alph.deg()
    N = ((1<<deg)-1)//p
    nimber = alph**p # p divides order of alpha_p
    for div in divs(N):
        if nimber**div == Nim(1):
            return p*div

M = 50
N = 60
for p in small_primes[25:]:
    if p == 151: continue
    D = alpha(p).deg()
    if M < D <= N:
        try:
            print(f'order of alpha_{p} is {alpha_ord(p)}')
        except:
            continue

In [30]:
[alpha(127) ** k for k in divs(4398046511103)]

[w**w**2 + 1,
 w**(w**2*3) + w**(w**2*2) + w**w**2 + 1,
 w**(w**2*6) + w**(w**2*5) + w**(w**2*4) + w**(w**2*3) + w**(w**2*2) + w**w**2 + w,
 w**(w**2*2 + 1) + w**(w**2*2) + w**(w**2 + 1) + 1,
 w**(w**2*6 + 2) + w**(w**2*6) + w**(w**2*5) + w**(w**2*4) + w**(w**2*3 + 2) + w**(w**2*3) + w**(w**2*2 + 2) + w**(w**2*2) + w**w**2 + w**2 + w + 2,
 w**(w**2*6 + 2)*2 + w**(w**2*6 + 1) + w**(w**2*5 + 2)*2 + w**(w**2*5 + 1) + w**(w**2*4 + 1)*3 + w**(w**2*3 + 1) + w**(w**2*2 + 1) + w**(w**2 + 2) + w**(w**2 + 1)*3 + w**w**2*2 + w**2*3 + w + 2,
 w**(w**2*6 + 2) + w**(w**2*6 + 1)*2 + w**(w**2*6)*2 + w**(w**2*5 + 1)*2 + w**(w**2*5) + w**(w**2*4 + 1)*2 + w**(w**2*4) + w**(w**2*3 + 2) + w**(w**2*3) + w**(w**2*2 + 2) + w**(w**2*2) + w**w**2 + w**2*3 + 1,
 w**(w**2*6 + 2)*3 + w**(w**2*6 + 1)*3 + w**(w**2*6) + w**(w**2*5 + 2)*3 + w**(w**2*5 + 1)*3 + w**(w**2*5) + w**(w**2*4 + 2)*3 + w**(w**2*4 + 1)*3 + w**(w**2*4) + w**(w**2*3 + 2)*3 + w**(w**2*3 + 1)*3 + w**(w**2*3) + w**(w**2*2 + 2)*3 + w**(w**2*2 + 1)*3 

In [ ]:
p=73
y = kappa(p) ** 13797

sorted([(k, y ** k) for k in range(1, p+1)], key=lambda x:x[1].val)

In [164]:
from itertools import product

def nim_iter(coeff, basis, dim):
    vec1 = np.array(basis[::-1], dtype=object)
    for coeffs in product(coeff, repeat=dim):
        vec2 = np.array(coeffs, dtype=object)
        yield np.dot(vec1, vec2)
        
d = 7
N = 42
good_ords = [div for div in divs(2**N-1) if div % (2**d - 1)==0]
F64 = nim_iter([Nim(k) for k in range(4)], [kappa(3)**j for j in range(3)], 3)
# c = [n for n in F64]
b = [kappa(9)**n for n in range(3)]

        

In [45]:
y = Nim(W(W(2))+1) ** (89756051247 // 127)
sorted([(n, y**n) for n in range(128)], key=lambda z:z[1].val)

[(0, 1),
 (127, 1),
 (30,
  w**(w**2*4 + 1) + w**(w**2*2 + 2)*2 + w**(w**2*2 + 1) + w**(w**2*2) + w**(w**2 + 2) + w**(w**2 + 1)*3 + w**w**2*2),
 (41,
  w**(w**2*4 + 1) + w**(w**2*2 + 2)*2 + w**(w**2*2 + 1) + w**(w**2*2) + w**(w**2 + 2) + w**(w**2 + 1)*3 + w**w**2*2 + 1),
 (71,
  w**(w**2*4 + 2) + w**(w**2*4) + w**(w**2*2 + 1)*3 + w**(w**2*2)*2 + w**(w**2 + 1)*3),
 (42,
  w**(w**2*4 + 2) + w**(w**2*4) + w**(w**2*2 + 1)*3 + w**(w**2*2)*2 + w**(w**2 + 1)*3 + 1),
 (60,
  w**(w**2*4 + 2) + w**(w**2*4 + 1) + w**(w**2*4) + w**(w**2*2 + 2)*2 + w**(w**2*2 + 1)*2 + w**(w**2*2)*3 + w**(w**2 + 2) + w**w**2*2),
 (82,
  w**(w**2*4 + 2) + w**(w**2*4 + 1) + w**(w**2*4) + w**(w**2*2 + 2)*2 + w**(w**2*2 + 1)*2 + w**(w**2*2)*3 + w**(w**2 + 2) + w**w**2*2 + 1),
 (15,
  w**(w**2*4 + 2)*2 + w**(w**2*4)*3 + w**(w**2*2 + 2)*2 + w**(w**2 + 2)*2 + w**(w**2 + 1)*3 + w**w**2),
 (84,
  w**(w**2*4 + 2)*2 + w**(w**2*4)*3 + w**(w**2*2 + 2)*2 + w**(w**2 + 2)*2 + w**(w**2 + 1)*3 + w**w**2 + 1),
 (99,
  w**(w**2*4 + 2)*

In [11]:
def denom_deg(nimber:Nim):
    if nimber.val < Ordinal():
        return Nim(nimber.base >> 1).deg()
    coeff = nimber.as_vec(as_nim=True)
    baseF = Nim(nimber.base_field())
    p, n = prime_power(baseF.is_field())
    coeff.append(baseF**p)
    return lcm(*[c.deg() for c in coeff])
    
# denom_deg(Nim(W()+4))
# nimber = Nim(W(W(2))+W(13)*15+2)
# N = nimber.deg()
# D = denom_deg(nimber)
# nimber.det() == nimber**((2**N-1)//(2**D-1))
# N, D
def cascade(nimber:Nim):
    stats= []
    n = 0
    while nimber.val > 1:
        N = nimber.deg()
        D = denom_deg(nimber)
        stats.append([n, nimber, N, D])
        nimber = nimber.det()
        n += 1
    stats.append([n, 1, 1, 1])
    return stats

cascade(kappa(f(47))+Nim(1))
        

[[0, w**w**7 + 1, 5060, 220],
 [1, w**w**3, 220, 20],
 [2, w**w + 1, 20, 4],
 [3, 5, 4, 2],
 [4, 2, 2, 1],
 [5, 1, 1, 1]]

In [ ]:
x = Nim(W(2)+W()*3)
y = kappa(101)+Nim(257)
Nim(W(2)+12345).inv()
x.det()

1

In [ ]:
kappa_deg(small_primes[13])

1806

In [ ]:
g = Nim(W(2)+W()*3)
g**3 + g

1

In [ ]:
from pprint import pprint

In [ ]:
X = np.array([[1, 2], [3, 4]])
pprint(X.T)

array([[1, 3],
       [2, 4]])


In [ ]:
x = W(W(1, 4), 6) + W(W(1, 2, 8), 7) + W(3, 1, 16)
y = W(26,3)+W(17, 7)+W(10)+W()+1234
z = 12345
Nim(x).as_vec()

[w**3 + 16, 0, w**8*7, 0, 6]

In [ ]:
W(W())**3

w**(w*3)

In [ ]:
o = Nim(W(W(7)))
o2 = (o**10000).val
len(ord_decomp(o2))

55

In [ ]:
def ord_decomp_iter(ordinal):
    ''' Yields inf_1, inf_2, ..., inf_n, finite term one by one '''
    if isinstance(ordinal, int):
        yield ordinal
        return

    # Yield the highest term first
    yield Ordinal(ordinal.exponent, ordinal.coefficient)

    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        yield Ordinal(remainder.exponent, remainder.coefficient, 0)
        remainder = remainder.addend

    assert isinstance(remainder, int)
    yield remainder

In [ ]:
F = Nim((1<<(1<<5))+6)
F.order(), F**(F.order()), F.is_gen() # wow! so fast if happens to be generator

(18446744073709551615, 1, True)

In [ ]:
def least_gen(L):
    num = 1<<(1<<L)
    while not Nim(num).is_gen():
        num+=1
    return num

In [ ]:
def power_omega(nimber):
    order = nimber.order()
    powers = {nimber**n for n in range(order)}
    possible = {Nim(n) for n in range(1, nimber.field+1)}
    excludant = possible ^ powers
    return min(excludant, key=lambda x:x.val)
    

In [ ]:
[least_gen(L) for L in range(7)]

[2, 4, 18, 258, 65540, 4294967302, 18446744073709551618]

In [ ]:
a = W(W())
x = Nim(a**2+a+1)
g = x**(2**10+1) # generator of GF(2^10)
g

w**(w*4)*15 + w**(w*3)*5 + w**(w*2) + w**w*2

### **CONJECTURE**
$\alpha\in 2^{2^n}$ is generator $\Longleftrightarrow$ $\det_*(\alpha)=n$

Not quite... ==> is true, but <== can fail for n > 5

In [ ]:
# # use dynamic programming for the recursive functions
# f_p = {}
# k_p = {}
# k_deg = {}
# Q_p = {}
# excess_p = {p: ex for (p, ex) in zip(small_primes, excess)}

NameError: name 'excess' is not defined

In [ ]:
from Z import is_prime, divs, prime_power, factor, gcd, lcm


def lcm(*args):
    if len(args) == 0: return 0
    elif len(args) == 1: return args[0]
    else:
        n, m, *rest = args
    mult = n*m // gcd(n, m)[-1]
    for num in rest:
        mult = mult * num // gcd(mult, num)[-1]
    return mult

def pi(n : int) -> int:
    ''' prime counting function: returns # primes < n'''
    with open('small_primes.txt') as file:
        num_less = 0
        for line in file:
            p = int(line)
            if p < n:
                num_less += 1
            else:
                return num_less
        raise ValueError('n is too big to calculate pi(n) by brute force')
            
            
def f(p):
    '''Lenstra's f function: 
    for prime p, f(p) is min{ n | p divides 2^n-1}
    It is always the case that f(p) divides p-1'''
    assert(is_prime(p)), 'f(p) in only defined for primes p'
    if p in f_p:
        return f_p[p]
    divisors = divs(p-1)
    for div in divisors[:-1]:
        if ((1<<div) -1) % p == 0:
            f_p[p] = div
            return div
    f_p[p] = divisors[-1]
    return divisors[-1]

def kappa(h : int) -> Nim:
    assert h > 0
    if h in k_p:
        return k_p[h]
    if h == 1 : result = Nim(0)
    elif prime_power(h):
        p, n = prime_power(h)
        k = pi(p)
        exp = Ordinal(k, h//p) if k > 0 else h//p
        result = Nim(2**exp)
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = kappa(g)
        else:
            result = kappa(q) + kappa(g)
    k_p[h] = result
    return result
        
def Q(h : int) -> list[int]:
    ''' kappa(h) = sum_{q in Q} kappa(q), q prime powers'''
    assert h > 0
    if h in Q_p:
        return Q_p[h]
    if h == 1 : 
        result = []
    elif prime_power(h):
        result = [h]
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = Q(g)
        else:
            result = Q(g) + Q(q)
    Q_p[h] = result
    return result
def kappa_deg(h : int) -> int:
    ''' faster way to calculate degree for kappa_h'''
    if h in k_deg:
        return k_deg[h]
    if h == 1: result = 1
    
    elif prime_power(h):
        p, n = prime_power(h)
        if p == 2: result = h # deg(k_{2^n}) = 2^n
        else:
            result = h * lcm(kappa_deg(f(p)), (alpha(p) + kappa(f(p))).deg())
    else:
        result = lcm(*[kappa_deg(q) for q in Q(h)])
    k_deg[h] = result
    return result
    
def alpha(p : int) -> Nim:
    assert is_prime(p)
    if p in alpha_p:
        return Nim(alpha_p[p])
    else:
        if p in excess_p:
            alphaP = kappa(f(p)).val + excess_p[p]
            alpha_p[p] = alphaP
            return Nim(alphaP)
        d = kappa_deg(f(p))
        mersenne = ((1<<d) - 1)
        excess = 0
        # save some iterations with known lower bound for excess
        if len(Q(f(p))) == 1 and Q(f(p))[0] % 2 == 1:
            excess = 1 # Q(f(p)) = {q} odd prime power ==> excess >= 1
        if f(p) % 2 == 0 and prime_power(f(p)//2):
                if prime_power(f(p)//2)[0] == 3:
                    excess = 4 #f(p) = 2*3^k, k>0 ==> excess >=4           
        beta = Nim(kappa(f(p)).val + excess)
        expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        while expr:
            excess += 1
            beta = Nim(kappa(f(p)).val + excess)
            d = lcm(kappa_deg(f(p)), Nim(excess).deg())
            mersenne = ((1<<d) - 1)
            expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        alpha_p[p] = beta.val
        return beta
    

In [ ]:
alpha(107)

w**w**14 + 1

In [ ]:
k5 = Nim(W(W()))
g5 = k5+k5**(1<<5)+k5**(1<<10)+k5**(1<<15)
g5

w**(w*4)*10 + w**(w*3)*9 + w**(w*2)*8 + w**w

In [ ]:
def tr_kappa(p):
    nimber = kappa(p)
    d = kappa_deg(p)
    result = nimber
    for q in range(1, d//p):
        nimber = nimber ** (1 << p)
        result = result + nimber
    return result

In [ ]:
import pandas as pd

In [ ]:
d = {'p':excess_p.keys(), 
     'kappa_p':[str(kappa(p)) for p in excess_p], 
     'alpha_p': [str(alpha(p)) for p in excess_p], 
     'excess' : [str(excess_p[p]) for p in excess_p],
     'f(p)':[f(p) for p in excess_p],
     'Q(f(p))': [Q(f(p)) for p in excess_p],
     'identity_p': [f'$\\left[{kappa(p)._repr_latex_()[1:-1]}\\right]^{{{p}}}={alpha(p)._repr_latex_()[1:]}'for p in excess_p]}
df = pd.DataFrame(d)
df.to_csv('alpha_p(1).csv', index=True)

In [ ]:
# to_paste = df.identity_p.map(lambda s: f'\\item {s}')
# to_paste.to_csv('to_paste_latex.csv', index=False)

In [ ]:
Nim(W(W(100)))**557

NameError: name 'Nim' is not defined

In [ ]:
w=W()
test = {}
for i in df.index:
    test[i] = eval(df.loc[i, 'alpha_p'])

In [ ]:
# let's compute higher alphas

# with open('small_primes.txt') as file:
#     for line in file:
        
#         p = int(line)
        
#         start = time.time()
#         alpha(p)
#         end = time.time()
#         alphaP = alpha_p[p]
#         df_p = pd.DataFrame({'p':p, 
#                             'kappa_p':[str(kappa(p))], 
#                             'alpha_P': [str(alpha_P[p])], 
#                             'excess' : [str(alpha(p)+kappa(f(p)))],
#                             'f(p)':[f(p)],
#                             'Q(f(p))': [Q(f(p))], 
#                             'compute_sec':end - start})
#         df = pd.concat([df, df_p], ignore_index=True)
#         df.to_csv('lenstra_nimber.csv', index=False)
#         print(f'alpha_{p}={alphaP}, was computed in {end - start}')
        
        # 47 takes too long :( my code for multiplication is too inefficient
        # also, the way factorizartion turns out yields **insane** exponent

In [ ]:
excess = [3,0,0,1,1,0,0,4,1,0,1,0,1,1,1,1,1,0,0,0,1,1,1,1,0,
 1,0,1,0,0,1,0,1,0,1,0,1,4,1,0,1,0,0,0,0,0,0,1,1,1,
 1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,0,
 1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,0,0,0,
 0,1,1,1,1,0] # use https://oeis.org/A380496/